# CA-Mixer vs MLP-Mixer on CIFAR-100
Single-file experiment for Google Colab / Kaggle (free-tier T4). Run cells top to bottom.

What it produces (saved in the results folder):
* `summary.csv`, `table_main.tex`, `table_resolution.csv`, `table_complexity.csv`
* `fig_training_curves.png`, `fig_resolution.png`, `fig_complexity.png`
* per-run JSON (resumable: finished runs are skipped if the session restarts)

Metrics: top-1/top-5 accuracy, test loss, ECE (calibration), clean train acc + gap,
params, FLOPs, throughput, peak train memory, train time, multi-seed mean +/- std,
Welch t-test, zero-shot resolution robustness, and cost scaling vs resolution.

In [ ]:
# Config (edit here)
QUICK = False          # True = smoke test (1 seed, 3 epochs) -> run this first!
USE_DRIVE = False      # Colab only: save results to Google Drive so they survive disconnects
RUN_ABLATION = False   # True = extra CA-Mixer ablations (1 seed each, ~+35 min on T4)
RUN_RESOLUTION = True  # zero-shot test at other resolutions
RES_LIST = [16, 24, 32, 40, 48, 64]          # must be multiples of patch size (4)
COMPLEXITY_RES = [32, 64, 96, 128]

CFG = dict(
    epochs=30, bs=256, lr=1e-3, wd=0.05, warmup=2, ls=0.1, clip=1.0,
    seeds=[0, 1, 2],
    patch=4, dim=128, depth=6,
    mixer_token_hidden=256, mixer_channel_hidden=512,
    ca_hidden=128, ca_channel_hidden=512, fire_rate=0.8,
    amp=True,
)
if QUICK:
    CFG.update(epochs=3, seeds=[0])
    RES_LIST = [24, 32, 48]
    COMPLEXITY_RES = [32, 64]

In [ ]:
# Imports & environment
import os, sys, time, json, math, random, shutil
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if dev.type != "cuda":
    print("WARNING: no GPU found. Colab: Runtime > Change runtime type > T4 GPU. "
          "Kaggle: Settings > Accelerator > GPU T4 x2 (or P100).")
AMP = bool(CFG["amp"]) and dev.type == "cuda"
torch.backends.cudnn.benchmark = True

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/ca_mixer_results"
elif IN_KAGGLE:
    OUT = "/kaggle/working/ca_mixer_results"
elif IN_COLAB:
    OUT = "/content/ca_mixer_results"
else:
    OUT = "./ca_mixer_results"
RUNS = os.path.join(OUT, "runs"); CKPT = os.path.join(OUT, "ckpt")
for d in (OUT, RUNS, CKPT):
    os.makedirs(d, exist_ok=True)
DATA_DIR = "/kaggle/working/data" if IN_KAGGLE else ("/content/data" if IN_COLAB else "./data")
print("torch", torch.__version__, "| device", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "")
print("results ->", OUT)

In [ ]:
# Models
# ---------------- MLP-Mixer baseline ----------------
class MixerBlock(nn.Module):
    def __init__(s, n, d, dt, dc):
        super().__init__()
        s.n1 = nn.LayerNorm(d); s.t1 = nn.Linear(n, dt); s.t2 = nn.Linear(dt, n)
        s.n2 = nn.LayerNorm(d); s.c1 = nn.Linear(d, dc); s.c2 = nn.Linear(dc, d)
    def forward(s, x):                                   # x: B,N,D
        y = s.n1(x).transpose(1, 2)
        x = x + s.t2(F.gelu(s.t1(y))).transpose(1, 2)    # token mixing (fixed N x N-style weights)
        return x + s.c2(F.gelu(s.c1(s.n2(x))))           # channel mixing

class MLPMixer(nn.Module):
    def __init__(s, img=32, patch=4, dim=128, depth=6, dt=256, dc=512, nc=100):
        super().__init__()
        s.embed = nn.Conv2d(3, dim, patch, patch)
        n = (img // patch) ** 2
        s.blocks = nn.Sequential(*[MixerBlock(n, dim, dt, dc) for _ in range(depth)])
        s.norm = nn.LayerNorm(dim); s.head = nn.Linear(dim, nc)
    def forward(s, x):
        x = s.embed(x).flatten(2).transpose(1, 2)
        return s.head(s.norm(s.blocks(x)).mean(1))

# ---------------- CA-Mixer ----------------
class LN2d(nn.Module):
    """Per-position LayerNorm over channels (no cross-position statistics -> resolution agnostic)."""
    def __init__(s, c):
        super().__init__(); s.ln = nn.LayerNorm(c)
    def forward(s, x):
        return s.ln(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

def perception_kernels():
    sx = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]])
    sy = torch.tensor([[-1., -2, -1], [0, 0, 0], [1, 2, 1]])
    lp = torch.tensor([[0., 1, 0], [1, -4, 1], [0, 1, 0]])
    return torch.stack([sx, sy, lp])                      # 3,3,3

class CATokenMixer(nn.Module):
    def __init__(s, dim, hidden, fire_rate=0.8, t_mode="adaptive", learn_filters=False):
        super().__init__()
        s.dim, s.fire, s.t_mode = dim, fire_rate, t_mode
        w = perception_kernels().repeat(dim, 1, 1).unsqueeze(1)   # (3*dim,1,3,3), groups=dim
        s.w = nn.Parameter(w, requires_grad=learn_filters)        # frozen by default
        s.f1 = nn.Conv2d(3 * dim, hidden, 1)                      # shared MLP f_theta (1x1 convs)
        s.f2 = nn.Conv2d(hidden, dim, 1)
        nn.init.zeros_(s.f2.weight); nn.init.zeros_(s.f2.bias)    # start as identity flow
    def steps(s, x):
        return max(x.shape[-2:]) if s.t_mode == "adaptive" else int(s.t_mode)
    def forward(s, h):
        for _ in range(s.steps(h)):
            p = F.conv2d(F.pad(h, (1, 1, 1, 1), mode="replicate"), s.w, groups=s.dim)
            d = torch.tanh(s.f2(F.gelu(s.f1(p))))
            if s.fire < 1.0:
                if s.training:
                    m = (torch.rand(h.size(0), 1, h.size(2), h.size(3), device=h.device) < s.fire).to(d.dtype)
                    d = d * m
                else:
                    d = d * s.fire            # expectation of the mask -> deterministic inference
            h = h + d
        return h

class CABlock(nn.Module):
    def __init__(s, dim, ca_hidden, dc, **kw):
        super().__init__()
        s.n1 = LN2d(dim); s.mix = CATokenMixer(dim, ca_hidden, **kw)
        s.n2 = LN2d(dim); s.c1 = nn.Conv2d(dim, dc, 1); s.c2 = nn.Conv2d(dc, dim, 1)
    def forward(s, x):
        n = s.n1(x)
        x = x + (s.mix(n) - n)                                    # outer residual around the NCA loop
        return x + s.c2(F.gelu(s.c1(s.n2(x))))

class CAMixer(nn.Module):
    def __init__(s, patch=4, dim=128, depth=6, ca_hidden=128, dc=512, nc=100, **kw):
        super().__init__()
        s.embed = nn.Conv2d(3, dim, patch, patch)
        s.blocks = nn.ModuleList([CABlock(dim, ca_hidden, dc, **kw) for _ in range(depth)])
        s.norm = LN2d(dim); s.head = nn.Linear(dim, nc)
    def forward(s, x):
        x = s.embed(x)
        for b in s.blocks:
            x = b(x)
        return s.head(s.norm(x).mean((2, 3)))                     # GAP -> any resolution

def build(kind, cfg, img=32, **kw):
    if kind == "mixer":
        return MLPMixer(img, cfg["patch"], cfg["dim"], cfg["depth"],
                        cfg["mixer_token_hidden"], cfg["mixer_channel_hidden"])
    kw.setdefault("fire_rate", cfg["fire_rate"])
    return CAMixer(cfg["patch"], cfg["dim"], cfg["depth"], cfg["ca_hidden"], cfg["ca_channel_hidden"], **kw)

def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
# Data (whole dataset lives on the GPU; augmentation done on GPU -> no CPU bottleneck)
def load_cifar100(root):
    tr = torchvision.datasets.CIFAR100(root, train=True, download=True)
    te = torchvision.datasets.CIFAR100(root, train=False, download=True)
    mean = torch.tensor([0.5071, 0.4865, 0.4409]).view(1, 3, 1, 1)
    std = torch.tensor([0.2673, 0.2564, 0.2762]).view(1, 3, 1, 1)
    def prep(ds):
        x = torch.from_numpy(ds.data).permute(0, 3, 1, 2).float().div(255)
        return ((x - mean) / std).to(dev), torch.tensor(ds.targets).to(dev)
    return prep(tr) + prep(te)

Xtr, ytr, Xte, yte = load_cifar100(DATA_DIR)
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))

def augment(x):
    """Random crop (pad 4, reflect) + horizontal flip, per-sample, on GPU."""
    B = x.size(0)
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i = torch.randint(0, 9, (B,), device=x.device); j = torch.randint(0, 9, (B,), device=x.device)
    ar = torch.arange(32, device=x.device)
    rows = (i[:, None] + ar)[:, :, None]; cols = (j[:, None] + ar)[:, None, :]
    x = x[torch.arange(B, device=x.device)[:, None, None], :, rows, cols].permute(0, 3, 1, 2)
    flip = torch.rand(B, device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x).contiguous()

In [ ]:
# Evaluation helpers
def resize(x, r):
    if x.shape[-1] == r:
        return x
    return F.interpolate(x, size=(r, r), mode="bilinear", align_corners=False, antialias=r < x.shape[-1])

@torch.no_grad()
def predict(model, X, res=32, back_to=None, bs=1000):
    model.eval(); outs = []
    for i in range(0, len(X), bs):
        xb = resize(X[i:i + bs], res)
        if back_to:
            xb = resize(xb, back_to)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            outs.append(model(xb).float())
    return torch.cat(outs)

def metrics(logits, y, nb=15):
    loss = F.cross_entropy(logits, y).item()
    conf, pred = logits.softmax(1).max(1); correct = (pred == y).float()
    top5 = (logits.topk(5, 1).indices == y[:, None]).any(1).float().mean().item() * 100
    edges = torch.linspace(0, 1, nb + 1, device=logits.device); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.float().mean().item() * abs(conf[m].mean().item() - correct[m].mean().item())
    return dict(loss=loss, top1=correct.mean().item() * 100, top5=top5, ece=ece * 100)

def count_flops(model, res=32):
    """Forward FLOPs per image (2*MACs convention), measured with PyTorch's FlopCounterMode."""
    try:
        from torch.utils.flop_counter import FlopCounterMode
        model.eval(); x = torch.randn(1, 3, res, res, device=dev)
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            model(x)
        return fc.get_total_flops() / 1e9
    except Exception as e:
        print("FLOP count failed:", e); return float("nan")

@torch.no_grad()
def bench(model, res=32, bs=512, warm=3, iters=10):
    """Inference throughput (img/s) and peak memory (GB), fp16 autocast."""
    model.eval(); x = torch.randn(bs, 3, res, res, device=dev)
    if dev.type == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
        for _ in range(warm): model(x)
        if dev.type == "cuda": torch.cuda.synchronize()
        t = time.time()
        for _ in range(iters): model(x)
        if dev.type == "cuda": torch.cuda.synchronize()
    dt = time.time() - t
    mem = torch.cuda.max_memory_allocated() / 1e9 if dev.type == "cuda" else float("nan")
    return bs * iters / dt, mem

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# Training
def run_experiment(label, kind, kw, seed, cfg):
    tag = label.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "").replace(",", "")
    path = os.path.join(RUNS, f"{tag}_s{seed}.json")
    if os.path.exists(path):
        print(f"[skip] {label} seed {seed} (found {path})"); return json.load(open(path))
    set_seed(seed)
    model = build(kind, cfg, **kw).to(dev)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([
        dict(params=[p for p in params if p.ndim > 1], weight_decay=cfg["wd"]),
        dict(params=[p for p in params if p.ndim <= 1], weight_decay=0.0)], lr=cfg["lr"])
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    N, bs, E = len(Xtr), cfg["bs"], cfg["epochs"]; spe = N // bs; total = E * spe; warm = cfg["warmup"] * spe
    lr_at = lambda s: cfg["lr"] * (s + 1) / warm if s < warm else \
        0.5 * cfg["lr"] * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    print(f"\n=== {label} | seed {seed} | params {n_params(model)/1e6:.3f}M ===")
    if dev.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    hist, step, t_train = [], 0, 0.0
    for ep in range(E):
        model.train(); t0 = time.time(); perm = torch.randperm(N, device=dev)
        tl = torch.zeros((), device=dev); tc = torch.zeros((), device=dev)
        for it in range(spe):
            idx = perm[it * bs:(it + 1) * bs]; xb, yb = augment(Xtr[idx]), ytr[idx]
            for g in opt.param_groups: g["lr"] = lr_at(step)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                out = model(xb); loss = F.cross_entropy(out, yb, label_smoothing=cfg["ls"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(params, cfg["clip"]); scaler.step(opt); scaler.update()
            tl += loss.detach() * len(idx); tc += (out.argmax(1) == yb).sum(); step += 1
        if dev.type == "cuda": torch.cuda.synchronize()
        et = time.time() - t0; t_train += et
        m = metrics(predict(model, Xte), yte)
        hist.append(dict(epoch=ep + 1, train_loss=tl.item() / (spe * bs), train_acc=tc.item() / (spe * bs) * 100,
                         test_loss=m["loss"], test_acc=m["top1"], time=et))
        print(f"ep {ep+1:02d}/{E} | loss {hist[-1]['train_loss']:.3f} | train {hist[-1]['train_acc']:.1f} | "
              f"test {m['top1']:.2f} | {et:.1f}s" + (f" | est. total {et*E/60:.1f} min" if ep == 0 else ""))
    peak = torch.cuda.max_memory_allocated() / 1e9 if dev.type == "cuda" else float("nan")
    # ---- final-epoch evaluation (no checkpoint selection on the test set) ----
    fm = metrics(predict(model, Xte), yte)
    tr_m = metrics(predict(model, Xtr[:10000]), ytr[:10000])
    thr, imem = bench(model, 32)
    res = dict(label=label, kind=kind, seed=seed, **{k: fm[k] for k in ("top1", "top5", "loss", "ece")},
               train_acc_clean=tr_m["top1"], gap=tr_m["top1"] - fm["top1"], params_M=n_params(model) / 1e6,
               gflops=count_flops(model, 32), train_min=t_train / 60, sec_per_epoch=t_train / E,
               throughput=thr, peak_train_mem_gb=peak, history=hist, res_native={}, res_adapt={})
    if RUN_RESOLUTION:
        for r in RES_LIST:
            try:
                res["res_native"][r] = metrics(predict(model, Xte, r), yte)["top1"]
            except Exception:                    # e.g. MLP-Mixer: token-mixing weights tied to N
                res["res_native"][r] = None
            res["res_adapt"][r] = metrics(predict(model, Xte, r, back_to=32), yte)["top1"]
    json.dump(res, open(path, "w")); torch.save(model.state_dict(), os.path.join(CKPT, f"{tag}_s{seed}.pt"))
    print(f"final: top1 {res['top1']:.2f} top5 {res['top5']:.2f} ECE {res['ece']:.2f} | {res['train_min']:.1f} min")
    del model, opt
    if dev.type == "cuda": torch.cuda.empty_cache()
    return res

MAIN = [("MLP-Mixer", "mixer", {}), ("CA-Mixer", "ca", {})]
ABL = [("CA-Mixer (T=4 fixed)", "ca", dict(t_mode=4)),
       ("CA-Mixer (no mask)", "ca", dict(fire_rate=1.0)),
       ("CA-Mixer (learnable filters)", "ca", dict(learn_filters=True))]

all_runs = []
for seed in CFG["seeds"]:                       # seed-major order -> balanced partial results
    for label, kind, kw in MAIN:
        all_runs.append(run_experiment(label, kind, kw, seed, CFG))
if RUN_ABLATION:
    for label, kind, kw in ABL:
        all_runs.append(run_experiment(label, kind, kw, CFG["seeds"][0], CFG))

In [ ]:
# Complexity scaling (random-init models, one per resolution; MLP-Mixer is rebuilt for each N)
def complexity_benchmark(cfg, resolutions, bs=32):
    rows = []
    for r in resolutions:
        for name, kind in [("MLP-Mixer", "mixer"), ("CA-Mixer", "ca")]:
            try:
                m = build(kind, cfg, img=r).to(dev).eval()
                thr, mem = bench(m, r, bs=bs, warm=2, iters=5)
                rows.append(dict(model=name, res=r, tokens=(r // cfg["patch"]) ** 2, params_M=n_params(m) / 1e6,
                                 gflops=count_flops(m, r), img_per_s=thr, infer_mem_gb=mem,
                                 T=(r // cfg["patch"]) if kind == "ca" else None))
                del m
            except Exception as e:                # OOM etc.
                print("benchmark failed", name, r, str(e)[:80]); rows.append(dict(model=name, res=r))
            if dev.type == "cuda": torch.cuda.empty_cache()
    return pd.DataFrame(rows)

cdf = complexity_benchmark(CFG, COMPLEXITY_RES)
cdf.to_csv(os.path.join(OUT, "table_complexity.csv"), index=False)
print(cdf.round(3).to_string(index=False))

In [ ]:
# Aggregate results -> tables
runs = [json.load(open(os.path.join(RUNS, f))) for f in sorted(os.listdir(RUNS)) if f.endswith(".json")]
df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("history", "res_native", "res_adapt")} for r in runs])
cols = [("top1", "Top-1 (%)", 2), ("top5", "Top-5 (%)", 2), ("loss", "Test loss", 3), ("ece", "ECE (%)", 2),
        ("gap", "Train-test gap", 2), ("params_M", "Params (M)", 3), ("gflops", "GFLOPs/img", 3),
        ("sec_per_epoch", "s/epoch", 1), ("train_min", "Train (min)", 1),
        ("throughput", "Img/s", 0), ("peak_train_mem_gb", "Train mem (GB)", 2)]
order = [l for l in ["MLP-Mixer", "CA-Mixer"] + [a[0] for a in ABL] if l in df.label.unique()]
rows = []
for l in order:
    g = df[df.label == l]; row = {"Model": l, "n_seeds": len(g)}
    for k, name, d in cols:
        row[name] = f"{g[k].mean():.{d}f} ± {g[k].std(ddof=1):.{d}f}" if len(g) > 1 else f"{g[k].mean():.{d}f}"
    rows.append(row)
summary = pd.DataFrame(rows); summary.to_csv(os.path.join(OUT, "summary.csv"), index=False)
print(summary.to_string(index=False))

# LaTeX table (no extra dependencies)
def to_tex(d):
    cs = "l" + "c" * (d.shape[1] - 1)
    L = [r"\begin{table}[t]\centering\small", r"\caption{CIFAR-100 results (mean $\pm$ std over seeds, final epoch).}",
         rf"\begin{{tabular}}{{{cs}}}", r"\toprule", " & ".join(d.columns).replace("%", r"\%") + r" \\", r"\midrule"]
    L += [" & ".join(str(v) for v in r).replace("±", r"$\pm$") + r" \\" for r in d.values]
    return "\n".join(L + [r"\bottomrule", r"\end{tabular}", r"\end{table}"])
open(os.path.join(OUT, "table_main.tex"), "w").write(to_tex(summary.drop(columns=["n_seeds"])))

# significance test (Welch t-test, CA-Mixer vs MLP-Mixer)
try:
    from scipy import stats
    a = df[df.label == "CA-Mixer"].top1.values; b = df[df.label == "MLP-Mixer"].top1.values
    if len(a) > 1 and len(b) > 1:
        t, p = stats.ttest_ind(a, b, equal_var=False)
        print(f"\nWelch t-test top-1: CA-Mixer {a.mean():.2f} vs MLP-Mixer {b.mean():.2f} | "
              f"diff {a.mean()-b.mean():+.2f} pts | t={t:.2f}, p={p:.4f}  (n={len(a)} seeds; low power)")
except Exception as e:
    print("t-test skipped:", e)

In [ ]:
# Resolution robustness table + figures
rr = []
for r in runs:
    for res_, v in r["res_native"].items():
        rr.append(dict(label=r["label"], seed=r["seed"], res=int(res_), mode="native", top1=v))
    for res_, v in r["res_adapt"].items():
        rr.append(dict(label=r["label"], seed=r["seed"], res=int(res_), mode="resize->32", top1=v))
rdf = pd.DataFrame(rr)
if len(rdf):
    rt = rdf.groupby(["label", "mode", "res"]).top1.agg(["mean", "std"]).reset_index()
    rt.to_csv(os.path.join(OUT, "table_resolution.csv"), index=False); print(rt.round(2).to_string(index=False))

def curves(label):
    H = [r["history"] for r in runs if r["label"] == label]
    return np.array([[h[k] for h in hh] for hh in H for k in ["test_acc"]]), \
           np.array([[h["train_loss"] for h in hh] for hh in H])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for l in [x for x in order if x in ("MLP-Mixer", "CA-Mixer")]:
    ta, tl_ = curves(l); ep = np.arange(1, ta.shape[1] + 1)
    for a_, v, name in ((ax[0], tl_, "Train loss"), (ax[1], ta, "Test top-1 (%)")):
        m, s = v.mean(0), v.std(0); a_.plot(ep, m, label=l); a_.fill_between(ep, m - s, m + s, alpha=.2); a_.set_ylabel(name)
for a_ in ax: a_.set_xlabel("Epoch"); a_.grid(alpha=.3); a_.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_training_curves.png"), dpi=200); plt.show()

if len(rdf):
    plt.figure(figsize=(6, 4))
    for (l, mode), g in rdf.dropna(subset=["top1"]).groupby(["label", "mode"]):
        if l not in ("MLP-Mixer", "CA-Mixer"): continue
        s = g.groupby("res").top1.agg(["mean", "std"]).fillna(0)
        plt.errorbar(s.index, s["mean"], yerr=s["std"], marker="o", capsize=3,
                     label=f"{l} ({'native' if mode=='native' else 'resize→32 adapter'})",
                     ls="-" if mode == "native" else "--")
    plt.xlabel("Test resolution (trained at 32×32)"); plt.ylabel("Top-1 (%)"); plt.grid(alpha=.3); plt.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_resolution.png"), dpi=200); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for name, g in cdf.dropna(subset=["gflops"]).groupby("model"):
    ax[0].plot(g.tokens, g.gflops, "o-", label=name); ax[1].plot(g.tokens, g.img_per_s, "o-", label=name)
    ax[2].plot(g.tokens, g.params_M, "o-", label=name)
for a_, t in zip(ax, ["GFLOPs / image", "Inference img/s (bs=32)", "Parameters (M)"]):
    a_.set_xscale("log"); a_.set_yscale("log"); a_.set_xlabel("Number of tokens N"); a_.set_title(t); a_.grid(alpha=.3); a_.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_complexity.png"), dpi=200); plt.show()

In [ ]:
# Zip results for download
zp = shutil.make_archive(os.path.join(os.path.dirname(OUT.rstrip("/")) or ".", "ca_mixer_results"), "zip", OUT)
print("Saved:", zp)
# Colab: uncomment to download
# from google.colab import files; files.download(zp)
# Kaggle: the zip/folder in /kaggle/working appears under the notebook's "Output" tab after "Save Version".